# Day 1 — PyTorch Fundamentals
### Your first deep learning implementation

**What we build today:**
- Understand PyTorch tensors (the building block of everything)
- Build a neural network from scratch
- Write a training loop and watch the model actually learn

**How to use this notebook:**
Run each cell top to bottom (Shift+Enter). Read the explanation before each code block. If a line is unclear, ask.

---

## Part 1 — Tensors

A **tensor** is just a multi-dimensional array of numbers. That's it.

- A single number → 0D tensor (scalar)
- A list of numbers → 1D tensor (vector)
- A grid of numbers → 2D tensor (matrix)
- A stack of grids → 3D tensor (e.g., an image: height × width × channels)

Every piece of data in deep learning — images, audio, text, spikes — becomes a tensor. Every weight in a neural network is a tensor. PyTorch's job is to do math on tensors very fast (on GPU if available).

In [ ]:
import torch

# Create a 1D tensor (a vector of 5 numbers)
a = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print("Tensor a:", a)
print("Shape:", a.shape)   # shape tells you the dimensions
print("dtype:", a.dtype)   # dtype tells you the number type (float32 is standard in DL)

In [ ]:
# A 2D tensor — think of it as a matrix (rows x columns)
# This could represent a mini dataset: 3 samples, each with 4 features
data = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],   # sample 1
    [5.0, 6.0, 7.0, 8.0],   # sample 2
    [9.0, 10.0, 11.0, 12.0] # sample 3
])
print("Shape:", data.shape)  # torch.Size([3, 4]) → 3 rows, 4 columns
print("First sample:", data[0])        # indexing a row
print("First feature of all samples:", data[:, 0])  # indexing a column

In [ ]:
# Tensor math works element-wise — PyTorch handles all the loops for you
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

print("Add:     ", x + y)        # [5, 7, 9]
print("Multiply:", x * y)        # [4, 10, 18]
print("Dot product:", torch.dot(x, y))  # 1*4 + 2*5 + 3*6 = 32

# Matrix multiplication — this is the core operation in every neural network layer
A = torch.ones(3, 4)   # 3x4 matrix of ones
B = torch.ones(4, 2)   # 4x2 matrix of ones
C = A @ B              # @ is matrix multiply — result is 3x2
print("Matrix multiply shape:", C.shape)

**Quick check — before moving on, make sure you can answer:**
- What does `shape` tell you?
- If I have a tensor of shape `[100, 28, 28]`, what could it represent? (Hint: 100 images, each 28×28 pixels)
- What does `@` do?

---

## Part 2 — How a Neural Network Actually Works in Code

You know conceptually: input → layers → output → loss → backprop → update weights.

Here's how that maps to PyTorch:

| Concept | PyTorch |
|---|---|
| A layer | `nn.Linear(in, out)` — does `y = Wx + b` |
| Activation function | `torch.relu()`, `torch.sigmoid()`, etc. |
| The full network | A class that inherits `nn.Module` |
| Loss function | `nn.MSELoss()`, `nn.CrossEntropyLoss()`, etc. |
| Optimizer (updates weights) | `torch.optim.Adam(...)` |
| One forward pass | `output = model(input)` |
| Backprop | `loss.backward()` |
| Weight update | `optimizer.step()` |

In [ ]:
import torch
import torch.nn as nn

# Every neural network in PyTorch is a class that inherits from nn.Module
class SimpleNet(nn.Module):
    
    def __init__(self):
        super().__init__()  # always call this — it sets up PyTorch internals
        
        # Define the layers
        # nn.Linear(in_features, out_features) → a fully connected layer
        # It learns a weight matrix W and bias b, and computes: output = W * input + b
        self.layer1 = nn.Linear(2, 4)   # 2 inputs → 4 hidden neurons
        self.layer2 = nn.Linear(4, 1)   # 4 hidden neurons → 1 output
    
    def forward(self, x):
        # This defines the forward pass — what happens when data flows through
        x = self.layer1(x)         # pass through layer 1
        x = torch.relu(x)          # apply ReLU activation (clips negatives to 0)
        x = self.layer2(x)         # pass through layer 2
        return x

# Create the network
model = SimpleNet()
print(model)  # shows the architecture

# Count total learnable parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal learnable parameters: {total_params}")
# layer1: 2*4 weights + 4 biases = 12
# layer2: 4*1 weights + 1 bias  = 5
# total = 17 parameters

In [ ]:
# Let's do a forward pass with a single input
# Our network expects 2 input values
sample_input = torch.tensor([[0.5, -0.3]])  # shape [1, 2] — 1 sample, 2 features

output = model(sample_input)
print("Input:", sample_input)
print("Output:", output)
print("Output shape:", output.shape)  # [1, 1] — 1 sample, 1 output value

# The output is currently random/meaningless — the weights are randomly initialized
# Training will adjust these weights to make the output meaningful

## Part 3 — Training the Network

**The problem we'll solve:** Given two numbers, predict their sum.

Simple? Yes. But the training loop is *identical* to what you'd use for a complex SNN.

**The training loop, every time:**
```
for each batch of data:
    1. Forward pass   → get prediction
    2. Compute loss   → how wrong were we?
    3. Backward pass  → compute gradients (backprop)
    4. Update weights → optimizer adjusts weights using gradients
    5. Zero gradients → reset for next batch (easy to forget this!)
```

In [ ]:
import torch
import torch.nn as nn

# ── 1. Generate training data ──────────────────────────────────────────────
# We want: output = input[0] + input[1]
# e.g. input [0.3, 0.7] → target 1.0

torch.manual_seed(42)  # for reproducibility — same random numbers every run

n_samples = 500
X = torch.rand(n_samples, 2)        # 500 random pairs of numbers between 0 and 1
y = X[:, 0:1] + X[:, 1:2]          # target = sum of the two inputs

print("X shape:", X.shape)   # [500, 2]
print("y shape:", y.shape)   # [500, 1]
print("\nFirst 3 samples:")
for i in range(3):
    print(f"  Input: {X[i].tolist()}  →  Target: {y[i].item():.4f}")

In [ ]:
# ── 2. Define model, loss, optimizer ───────────────────────────────────────

model = SimpleNet()

# Loss function: Mean Squared Error
# MSE = average of (prediction - target)²
# The closer to 0, the better our predictions
loss_fn = nn.MSELoss()

# Optimizer: Adam
# Adam adjusts the weights after each batch to reduce the loss
# lr = learning rate — how big a step to take each update
# Too large → unstable training. Too small → very slow.
# 0.01 is a reasonable default to start with.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ── 3. Training loop ───────────────────────────────────────────────────────

n_epochs = 200  # one epoch = one full pass through the training data

for epoch in range(n_epochs):
    
    # --- Forward pass ---
    predictions = model(X)           # run all 500 samples through the network
    loss = loss_fn(predictions, y)   # compare predictions to targets
    
    # --- Backward pass ---
    optimizer.zero_grad()   # IMPORTANT: clear old gradients (they accumulate otherwise)
    loss.backward()         # compute gradients via backprop
    optimizer.step()        # update weights using the gradients
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.6f}")

print("\nTraining complete!")

In [ ]:
# ── 4. Test the trained model ──────────────────────────────────────────────

# torch.no_grad() tells PyTorch not to track gradients during evaluation
# — we're not training, just testing, so no need for backprop
with torch.no_grad():
    test_inputs = torch.tensor([
        [0.2, 0.3],   # expected: 0.5
        [0.8, 0.1],   # expected: 0.9
        [0.5, 0.5],   # expected: 1.0
    ])
    test_preds = model(test_inputs)
    
    print("Input          | Expected | Predicted")
    print("-" * 45)
    for i in range(len(test_inputs)):
        inp = test_inputs[i].tolist()
        expected = inp[0] + inp[1]
        predicted = test_preds[i].item()
        print(f"{inp}  |  {expected:.2f}    |  {predicted:.4f}")

## Part 4 — What Just Happened? (The Key Ideas)

You just trained a neural network. Here's what each part was doing:

**`optimizer.zero_grad()`** — PyTorch accumulates gradients by default. If you forget this, gradients from previous batches pile up and your training goes wrong. Always zero gradients at the start of each step.

**`loss.backward()`** — This is backpropagation. PyTorch automatically computes "how much does each weight contribute to the loss?" (the gradient). You never have to write this by hand.

**`optimizer.step()`** — Uses the gradients to nudge every weight in the direction that reduces loss. Adam is a smart optimizer that adapts the step size per parameter — it's the default choice for most projects.

**The loss going down** — This is the model learning. Each epoch, weights get slightly better at predicting sums.

---

## Part 5 — Overfitting vs Underfitting (Explained Through Code)

These are the two failure modes of any trained model:

- **Underfitting** — the model is too simple, or not trained long enough. It performs badly on both training data AND new data.
- **Overfitting** — the model has memorized the training data but can't generalize. It performs great on training data but badly on new data.

The fix: always evaluate on data the model has **never seen** during training — a **validation set**.

In [ ]:
# The right way: split data into training and validation sets

torch.manual_seed(42)
n_samples = 500
X = torch.rand(n_samples, 2)
y = X[:, 0:1] + X[:, 1:2]

# 80% train, 20% validation — a very common split
split = int(0.8 * n_samples)   # = 400
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

In [ ]:
model2 = SimpleNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.01)

n_epochs = 200

for epoch in range(n_epochs):
    
    # --- Training step (model learns here) ---
    model2.train()                          # puts model in training mode
    preds_train = model2(X_train)
    train_loss = loss_fn(preds_train, y_train)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
    # --- Validation step (just checking — no learning) ---
    model2.eval()                           # puts model in eval mode (disables dropout etc.)
    with torch.no_grad():                   # no gradient tracking needed
        preds_val = model2(X_val)
        val_loss = loss_fn(preds_val, y_val)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss.item():.6f} | Val Loss: {val_loss.item():.6f}")

# What to watch for:
# GOOD:      both losses go down together
# OVERFIT:   train loss keeps dropping, val loss starts going UP
# UNDERFIT:  both losses are high and barely move

## Part 6 — The Optimizer (What Adam Actually Does)

An optimizer's job: use gradients to update weights to reduce loss.

**SGD (Stochastic Gradient Descent)** — the simplest optimizer. Steps in the direction of the gradient by a fixed amount (the learning rate). Slow and sensitive to learning rate choice.

**Adam** — adaptive learning rates per parameter. Generally converges faster and is more forgiving. It's the safe default for most projects including SNNs.

**The learning rate** is your most important hyperparameter:

| Learning Rate | Effect |
|---|---|
| Too high (e.g. 1.0) | Loss explodes or oscillates, model doesn't converge |
| Too low (e.g. 0.00001) | Training works but takes forever |
| Just right (e.g. 0.001–0.01) | Loss decreases steadily |

Rule of thumb: **start with `lr=0.001` for Adam**. It's almost always a reasonable starting point.

In [ ]:
# Let's see what happens with different learning rates

def train_with_lr(lr, n_epochs=100):
    torch.manual_seed(0)
    m = SimpleNet()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(n_epochs):
        pred = m(X_train)
        loss = loss_fn(pred, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()

for lr in [0.1, 0.01, 0.001, 0.0001]:
    final_loss = train_with_lr(lr)
    print(f"lr={lr:.4f} → Final loss: {final_loss:.6f}")

---

## Day 1 Summary — What You Now Know

| Concept | What it is |
|---|---|
| Tensor | Multi-dimensional array. Everything in DL is a tensor. |
| `nn.Module` | The base class for every neural network in PyTorch |
| `nn.Linear` | A fully connected layer: computes `y = Wx + b` |
| `forward()` | Defines how data flows through your network |
| `loss.backward()` | Backpropagation — computes gradients automatically |
| `optimizer.step()` | Updates weights using the gradients |
| `optimizer.zero_grad()` | Clears old gradients before each step |
| Learning rate | Controls step size. Start at 0.001 with Adam. |
| Train vs Val loss | How you detect overfitting and underfitting |

---

## Day 2 Preview

Tomorrow: **What makes a spiking neuron different**, the Leaky Integrate-and-Fire (LIF) model, and your first spiking neuron simulation.

Before then — try to answer these without looking:
1. What does `optimizer.zero_grad()` do and why is it necessary?
2. How would you know if your model is overfitting?
3. If train loss = 0.001 and val loss = 0.8, what's happening?
4. What does `with torch.no_grad():` do and when do you use it?